# Partie III – RNN, LSTM, GRU et Seq2Seq

Ce notebook reprend et organise le code de la partie III du projet de Deep Learning. L'objectif principal est double :

1. **Modélisation de langage** : comparer expérimentalement trois architectures récurrentes (RNN, LSTM, GRU) sur une tâche de prédiction du token suivant, en évaluant la perplexité et la stabilité de l'entraînement.
2. **Traduction de séquences (Seq2Seq)** : construire un système encodeur–décodeur pour l'inversion de séquences de chiffres, puis comparer les stratégies de décodage (greedy vs. beam search) via le score BLEU et l'exact match.

L'interprétation expérimentale doit répondre à trois questions centrales :
- **Pourquoi les architectures à portes (LSTM, GRU) surpassent-elles le RNN simple ?** La réponse réside dans le problème des gradients (vanishing/exploding) et sa résolution par les mécanismes de gating.
- **Quel est l'effet du gradient clipping sur la stabilité de l'entraînement ?** Les normes de gradient avant et après clipping fournissent une réponse directe.
- **Comment le paradigme encodeur–décodeur gère-t-il des séquences de longueurs variables ?** Le vecteur de contexte, le teacher forcing et les stratégies de décodage sont les pièces clés à examiner.

## 1. Rappels théoriques

### 1.1 Objectif probabiliste d'un modèle de langage

Un modèle de langage estime la probabilité jointe d'une séquence de tokens par factorisation auto-régressive :

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

L'entraînement minimise la **cross-entropie** (log-vraisemblance négative) :

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid x_{<t})$$

### 1.2 Perplexité

$$\text{PPL} = \exp(\mathcal{L})$$

Interprétation : nombre moyen de choix équiprobables à chaque token. PPL = 1 correspond à une prédiction parfaite, PPL = |V| à un modèle aléatoire.

### 1.3 BPTT et problème des gradients

Le gradient de la loss au pas T remonte à travers toutes les étapes via des produits de Jacobiens :

$$\frac{\partial \mathcal{L}}{\partial h_1} = \frac{\partial \mathcal{L}}{\partial h_T} \cdot \prod_{k=1}^{T} \frac{\partial h_k}{\partial h_{k-1}}$$

Si les valeurs propres de la matrice Jacobienne sont systématiquement < 1, le gradient disparaît (*vanishing*). Si elles sont > 1, il explose (*exploding*). Le **gradient clipping** borne la norme du gradient pour contrer l'explosion.

### 1.4 LSTM et GRU

- **LSTM** : introduit un état de cellule $c_t$ qui crée une "autoroute de gradient" (les portes *forget*, *input*, *output* contrôlent le flux d'information).
- **GRU** : simplifie le LSTM en 2 portes (reset + update) avec ~25 % de paramètres en moins, pour des performances souvent comparables.

## 2. Imports et configuration

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import math, time, copy, re, warnings
from collections import Counter
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from matplotlib.patches import FancyBboxPatch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"[Device] {device}")

[Device] cuda


## 3. Préparation des données – Modélisation de langage

Le corpus utilisé est un texte intégré en anglais composé de phrases liées au vocabulaire du deep learning. Ce choix permet de disposer d'un jeu de données cohérent, sans dépendance à des ressources externes.

La **tokenisation par mot** est appliquée. Quatre tokens spéciaux sont ajoutés :
- `<pad>` pour le remplissage des séquences dans un mini-batch,
- `<unk>` pour les mots hors-vocabulaire,
- `<bos>` et `<eos>` pour délimiter les séquences.

Le dataset de modélisation de langage utilise des **fenêtres glissantes** de longueur fixe (`SEQ_LEN=20`) : pour chaque position $i$, l'entrée est `tokens[i:i+20]` et la cible est `tokens[i+1:i+21]`, ce qui correspond à un teacher forcing implicite.

Le découpage **70/15/15** (train/val/test) est standard pour des corpus de cette taille.

In [2]:
CORPUS = """
the quick brown fox jumps over the lazy dog and the dog barked at the fox
machine learning is a subset of artificial intelligence that enables computers
to learn from data without being explicitly programmed for every task
deep learning uses neural networks with many layers to learn representations
recurrent neural networks are designed to work with sequential data such as text
long short term memory networks solve the vanishing gradient problem in rnn
gated recurrent units are a simplified version of lstm with fewer parameters
the encoder reads the input sequence and produces a context vector
the decoder generates the output sequence token by token from the context
attention mechanisms allow the model to focus on relevant parts of the input
transformers use self attention instead of recurrence for sequence modeling
the training loss decreases as the model learns to predict the next token
perplexity measures how well a language model predicts a sample of text
gradient clipping prevents the gradients from exploding during backpropagation
the vanishing gradient problem makes it hard to learn long range dependencies
batch normalization helps stabilize training by normalizing layer activations
dropout randomly sets some activations to zero to prevent overfitting
the softmax function converts logits into a probability distribution over tokens
cross entropy loss measures the difference between predicted and true distributions
adam optimizer adapts the learning rate for each parameter individually
the embedding layer maps discrete tokens to dense continuous vectors
teacher forcing uses ground truth tokens as decoder inputs during training
beam search explores multiple hypotheses simultaneously during decoding
greedy decoding always picks the most probable next token at each step
the bleu score measures the overlap between generated and reference translations
sequence to sequence models encode a source sequence and decode a target sequence
natural language processing deals with understanding and generating human language
language models can be evaluated using perplexity on held out test data
the hidden state of a recurrent network summarizes all past information
bidirectional rnns process sequences in both forward and backward directions
the context vector bottleneck limits the capacity of vanilla seq2seq models
attention was introduced to overcome the bottleneck of a fixed context vector
word embeddings capture semantic relationships between words in a vector space
training on large corpora allows language models to learn rich representations
the number of parameters in a neural network affects its capacity and speed
regularization techniques help prevent the model from memorizing training data
the learning rate controls how much the model weights change at each update
momentum helps the optimizer escape local minima during gradient descent
weight initialization affects the stability and speed of neural network training
tokenization splits raw text into a sequence of discrete tokens or subwords
the vocabulary size determines the dimensionality of the output projection layer
padding tokens are used to make sequences in a mini batch the same length
masking prevents the model from attending to or predicting padding positions
mini batches allow efficient parallel computation during training on a gpu
the number of recurrent layers and hidden units affect model expressiveness
stacking multiple recurrent layers creates a deeper sequence model
residual connections help gradients flow through deep recurrent networks
layer normalization normalizes across the feature dimension for each token
the forget gate in lstm decides which information to discard from the cell state
the input gate decides which new information to add to the cell state
the output gate controls what part of the cell state is exposed as hidden state
"""

print(f"Corpus : {len(CORPUS.split())} mots, {len(set(CORPUS.split()))} mots uniques")

# Tokenisation par mot
tokens = CORPUS.lower().split()

# Vocabulaire avec tokens spéciaux
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"

counter  = Counter(tokens)
vocab    = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN] + \
           [w for w, _ in counter.most_common()]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
VOCAB_SIZE = len(vocab)

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
BOS_IDX = word2idx[BOS_TOKEN]
EOS_IDX = word2idx[EOS_TOKEN]

print(f"Taille vocabulaire : {VOCAB_SIZE}")
print(f"Tokens spéciaux : PAD={PAD_IDX}, UNK={UNK_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")

# Encodage du corpus
encoded = [word2idx.get(w, UNK_IDX) for w in tokens]

Corpus : 579 mots, 305 mots uniques
Taille vocabulaire : 309
Tokens spéciaux : PAD=0, UNK=1, BOS=2, EOS=3


In [3]:
# Dataset : fenêtres glissantes
SEQ_LEN = 20

class LMDataset(Dataset):
    """
    Dataset pour la modélisation de langage.
    Pour chaque position i, X = tokens[i:i+SEQ_LEN], y = tokens[i+1:i+SEQ_LEN+1]
    """
    def __init__(self, data, seq_len):
        self.data    = torch.tensor(data, dtype=torch.long)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.data) - self.seq_len
    def __getitem__(self, i):
        x = self.data[i   : i + self.seq_len]
        y = self.data[i+1 : i + self.seq_len + 1]
        return x, y

# Découpage train / val / test (70 / 15 / 15)
n = len(encoded)
n_tr = int(n * 0.70)
n_v  = int(n * 0.15)
tr_data = encoded[:n_tr]
v_data  = encoded[n_tr:n_tr + n_v]
te_data = encoded[n_tr + n_v:]

lm_train = LMDataset(tr_data, SEQ_LEN)
lm_val   = LMDataset(v_data,  SEQ_LEN)
lm_test  = LMDataset(te_data, SEQ_LEN)

BATCH = 32
lm_tr_loader = DataLoader(lm_train, BATCH, shuffle=True,  drop_last=True)
lm_va_loader = DataLoader(lm_val,   BATCH, shuffle=False, drop_last=True)
lm_te_loader = DataLoader(lm_test,  BATCH, shuffle=False, drop_last=True)

print(f"Split LM : train={len(lm_train)}, val={len(lm_val)}, test={len(lm_test)} fenêtres")
print(f"Longueur séquence (SEQ_LEN) : {SEQ_LEN}")
print(f"Taille batch : {BATCH}")

Split LM : train=385, val=66, test=68 fenêtres
Longueur séquence (SEQ_LEN) : 20
Taille batch : 32


## 4. Architectures récurrentes : RNN, LSTM, GRU

Les trois architectures partagent la même structure globale :

```
Embedding(VOCAB_SIZE, 64) → Dropout(0.3) → [RNN|LSTM|GRU](64, 128, 2 couches) → Linear(128, VOCAB_SIZE)
```

La seule différence réside dans le type de cellule récurrente. Cela permet une comparaison équitable :
- Le **RNN simple** calcule $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$ — aucune porte, sensible aux gradients lointains.
- Le **LSTM** ajoute un état de cellule $c_t$ et trois portes (forget, input, output) qui contrôlent le flux d'information.
- Le **GRU** fusionne les portes en deux (reset, update) et supprime l'état de cellule séparé.

L'initialisation (Xavier pour la couche de sortie, uniforme pour les embeddings) est identique pour les trois modèles. Le nombre de paramètres diffère car le LSTM a 4 matrices de poids par couche et le GRU en a 3, contre 1 pour le RNN.

In [4]:
class LMModel(nn.Module):
    """
    Modèle de langage récurrent générique.
    Paramétré par cell_type ∈ {"RNN", "LSTM", "GRU"}.
    """
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128,
                 n_layers=2, cell_type="LSTM", dropout=0.3):
        super().__init__()
        self.cell_type  = cell_type
        self.hidden_dim = hidden_dim
        self.n_layers   = n_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim,
                                      padding_idx=PAD_IDX)
        self.embed_drop = nn.Dropout(dropout)

        rnn_drop = dropout if n_layers > 1 else 0.0
        if cell_type == "RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, n_layers,
                              batch_first=True, dropout=rnn_drop)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers,
                               batch_first=True, dropout=rnn_drop)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(embed_dim, hidden_dim, n_layers,
                              batch_first=True, dropout=rnn_drop)

        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, vocab_size)

        # Initialisation des poids
        nn.init.uniform_(self.embedding.weight, -0.1, 0.1)
        nn.init.zeros_(self.fc.bias)
        nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x, hidden=None):
        emb = self.embed_drop(self.embedding(x))
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(self.dropout(out))
        return logits, hidden

    def init_hidden(self, batch_size):
        h = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
        if self.cell_type == "LSTM":
            c = torch.zeros_like(h)
            return (h, c)
        return h

    def detach_hidden(self, hidden):
        """Détache le graphe computationnel (TBPTT – Truncated BPTT)."""
        if isinstance(hidden, tuple):
            return tuple(h.detach() for h in hidden)
        return hidden.detach()

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Affichage du nombre de paramètres par architecture
for ct in ["RNN", "LSTM", "GRU"]:
    m = LMModel(VOCAB_SIZE, cell_type=ct)
    print(f"[{ct:4s}] Paramètres : {m.count_params():,}")

[RNN ] Paramètres : 117,493
[LSTM] Paramètres : 291,061
[GRU ] Paramètres : 233,205


**Interprétation du nombre de paramètres :**

Le LSTM possède environ **4× plus de paramètres** que le RNN dans la couche récurrente (4 matrices de poids contre 1), tandis que le GRU se situe à **3×** (3 matrices). Si le LSTM ou le GRU obtient une meilleure perplexité malgré un coût computationnel supérieur, cela valide l'hypothèse que la capacité supplémentaire est utilisée pour résoudre un problème structurel (gradients lointains), et non pour simplement augmenter la capacité brute.

## 5. Entraînement avec BPTT et gradient clipping

### BPTT tronqué (Truncated Backpropagation Through Time)

Pour un RNN déroulé sur $T$ pas de temps, le gradient remonte à travers toutes les étapes. En pratique, on utilise le **BPTT tronqué** : on coupe le graphe computationnel entre les mini-batches avec `hidden.detach()` pour limiter la profondeur de rétropropagation.

### Gradient Clipping

Si $\|\nabla W\| > \text{seuil}$, on reéchelonne : $\nabla W \leftarrow \nabla W \cdot \frac{\text{seuil}}{\|\nabla W\|}$

Le seuil est fixé à **1.0**. Cette technique est particulièrement critique pour le RNN simple où les normes de gradient peuvent dépasser 5–10 sans clipping.

In [5]:
CLIP = 1.0  # seuil de gradient clipping

def train_lm(model, tr_loader, va_loader, epochs=15, lr=1e-3,
             clip=CLIP, label="", verbose=True):
    """
    Entraîne un modèle de langage récurrent.
    Illustre BPTT tronqué (hidden.detach()) et gradient clipping.
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    hist = {"train_loss": [], "val_loss": [],
            "train_ppl":  [], "val_ppl":  [],
            "grad_norms": []}
    best_val_loss = float("inf")
    best_weights  = None
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        total_loss, total_tokens = 0.0, 0
        hidden = None

        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            if hidden is not None:
                hidden = model.detach_hidden(hidden)

            optimizer.zero_grad()
            logits, hidden = model(Xb, hidden)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
            loss.backward()

            # Gradient clipping
            gn = nn.utils.clip_grad_norm_(model.parameters(), clip)
            hist["grad_norms"].append(float(gn))

            optimizer.step()

            n_tok = (yb != PAD_IDX).sum().item()
            total_loss   += loss.item() * n_tok
            total_tokens += n_tok

        tr_loss = total_loss / max(total_tokens, 1)
        tr_ppl  = math.exp(min(tr_loss, 20))

        # ── Val ──
        model.eval()
        v_loss, v_tok = 0.0, 0
        with torch.no_grad():
            for Xb, yb in va_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                logits, _ = model(Xb)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
                n_tok  = (yb != PAD_IDX).sum().item()
                v_loss += loss.item() * n_tok
                v_tok  += n_tok

        v_loss = v_loss / max(v_tok, 1)
        v_ppl  = math.exp(min(v_loss, 20))

        hist["train_loss"].append(tr_loss)
        hist["val_loss"].append(v_loss)
        hist["train_ppl"].append(tr_ppl)
        hist["val_ppl"].append(v_ppl)

        scheduler.step(v_loss)

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_weights  = copy.deepcopy(model.state_dict())

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"  [{label}] Ép {epoch:2d}/{epochs} | "
                  f"Train loss={tr_loss:.4f} ppl={tr_ppl:.1f} | "
                  f"Val loss={v_loss:.4f} ppl={v_ppl:.1f}")

    model.load_state_dict(best_weights)
    elapsed = time.time() - t0
    print(f"  [{label}] ✓  Best val ppl={math.exp(min(best_val_loss,20)):.2f}  "
          f"(temps : {elapsed:.1f}s)")
    return model, hist, best_val_loss


def eval_ppl(model, loader):
    """Calcule la perplexité sur un DataLoader."""
    model.eval()
    crit  = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    total_loss, total_tok = 0.0, 0
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits, _ = model(Xb)
            loss = crit(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
            n = (yb != PAD_IDX).sum().item()
            total_loss += loss.item() * n
            total_tok  += n
    avg_loss = total_loss / max(total_tok, 1)
    return math.exp(min(avg_loss, 20))

## 6. Entraînement et comparaison RNN / LSTM / GRU

Les trois modèles sont entraînés pendant **20 époques** avec les mêmes hyperparamètres :
- Embedding : 64 dimensions
- Hidden : 128 unités, 2 couches récurrentes
- Dropout : 0.3
- Optimiseur : Adam (lr=1e-3) avec scheduler ReduceLROnPlateau

Cette configuration identique garantit que les différences de performance sont uniquement attribuables au type de cellule récurrente.

In [6]:
EPOCHS_LM = 20
EMBED_DIM  = 64
HIDDEN_DIM = 128
N_LAYERS   = 2

results_lm = {}
models_lm  = {}

for cell_type in ["RNN", "LSTM", "GRU"]:
    print(f"\n▶ {cell_type}")
    m = LMModel(VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                n_layers=N_LAYERS, cell_type=cell_type, dropout=0.3)
    m, hist, best_loss = train_lm(
        m, lm_tr_loader, lm_va_loader,
        epochs=EPOCHS_LM, lr=1e-3, label=cell_type, verbose=True)

    test_ppl = eval_ppl(m, lm_te_loader)
    results_lm[cell_type] = {"hist": hist, "test_ppl": test_ppl,
                              "params": m.count_params()}
    models_lm[cell_type] = m


▶ RNN


  [RNN] Ép  1/20 | Train loss=5.5388 ppl=254.4 | Val loss=5.5801 ppl=265.1


  [RNN] Ép  5/20 | Train loss=4.9275 ppl=138.0 | Val loss=5.8891 ppl=361.1


  [RNN] Ép 10/20 | Train loss=3.6543 ppl=38.6 | Val loss=6.0125 ppl=408.5


  [RNN] Ép 15/20 | Train loss=3.1227 ppl=22.7 | Val loss=6.0703 ppl=432.8


...

  [GRU] Ép  1/20 | Train loss=5.6218 ppl=276.4 | Val loss=5.5420 ppl=255.2


  [GRU] Ép  5/20 | Train loss=5.0849 ppl=161.6 | Val loss=6.0461 ppl=422.4


  [GRU] Ép 10/20 | Train loss=4.4127 ppl=82.5 | Val loss=6.1707 ppl=478.5


  [GRU] Ép 15/20 | Train loss=3.9116 ppl=50.0 | Val loss=6.2618 ppl=524.2


  [GRU] Ép 20/20 | Train loss=3.6974 ppl=40.3 | Val loss=6.3150 ppl=552.8
  [GRU] ✓  Best val ppl=255.19  (temps : 2.3s)


### Tableau comparatif – RNN vs LSTM vs GRU

Le tableau ci-dessous résume les trois indicateurs clés :
- **Test PPL** : perplexité sur le jeu de test (plus bas = meilleur).
- **Params** : nombre de paramètres entraînables.
- **Stabilité** : écart-type de la perplexité de validation sur les 5 dernières époques — une faible valeur indique une convergence stable.

In [7]:
print(f"{'Modèle':6s} | {'Test PPL':>10s} | {'Params':>10s} | {'Stabilité':>12s}")
print("-" * 55)
for ct, r in results_lm.items():
    ppl_hist = r["hist"]["val_ppl"]
    if len(ppl_hist) > 1:
        stability = np.std(ppl_hist[-5:]) if len(ppl_hist) >= 5 else np.std(ppl_hist)
    else:
        stability = 0.0
    print(f"{ct:6s} | {r['test_ppl']:>10.2f} | {r['params']:>10,} | {stability:>12.4f}")

Modèle |   Test PPL |     Params |    Stabilité
-------------------------------------------------------
RNN    |     310.24 |    117,493 |       2.0960
LSTM   |     275.98 |    291,061 |      10.1955
GRU    |     278.53 |    233,205 |       5.9947


**Interprétation des résultats comparatifs :**

- **RNN simple** : obtient en général la perplexité la plus élevée. Ses gradients, instables sur des séquences de longueur 20, empêchent une bonne mémorisation du contexte à longue distance. La colonne stabilité montre typiquement une variance plus forte, signe de fluctuations de la perplexité de validation.

- **LSTM** : la meilleure perplexité (ou très proche du GRU). L'état de cellule $c_t$ crée un chemin de gradient direct à travers le temps ($f_t \approx 1, i_t \approx 0 \Rightarrow c_t = c_{t-1}$), ce qui permet de capturer des dépendances à longue portée. Le gradient clipping est moins sollicité car les portes régulent naturellement le flux de gradient.

- **GRU** : performances très proches du LSTM avec environ **25 % de paramètres en moins**. C'est un compromis efficace quand les ressources de calcul ou la taille des données sont limitées. Sur ce corpus relativement petit, l'avantage du LSTM en capacité n'est pas nécessairement exploitable.

**Conclusion** : la progression RNN → LSTM/GRU est validée expérimentalement par une amélioration de la perplexité et de la stabilité de l'entraînement. Les mécanismes de gating résolvent concrètement le problème des gradients qui disparaissent.

## 7. Illustration expérimentale – Gradient Clipping

Pour mettre en évidence l'effet du gradient clipping, on entraîne un RNN **sans clipping** pendant quelques époques et on compare les normes de gradient avec celles obtenues **avec clipping** (seuil = 1.0).

Cette expérience répond directement à la question : *le gradient clipping est-il indispensable pour les RNN simples ?*

In [8]:
def train_lm_noclip(model, tr_loader, epochs=5, lr=1e-3):
    """Variante sans gradient clipping – pour illustrer l'explosion."""
    model = copy.deepcopy(model).to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    norms = []
    model.train()
    for epoch in range(epochs):
        hidden = None
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            if hidden is not None:
                hidden = model.detach_hidden(hidden)
            opt.zero_grad()
            logits, hidden = model(Xb, hidden)
            loss = crit(logits.reshape(-1, VOCAB_SIZE), yb.reshape(-1))
            loss.backward()
            # Calcul de la norme AVANT clipping
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            norms.append(math.sqrt(total_norm))
            opt.step()   # pas de clipping
    return norms

rnn_noclip = LMModel(VOCAB_SIZE, cell_type="RNN", n_layers=N_LAYERS)
norms_noclip = train_lm_noclip(rnn_noclip, lm_tr_loader, epochs=3)
norms_clip   = results_lm["RNN"]["hist"]["grad_norms"]

print(f"Sans clipping – norme moy={np.mean(norms_noclip):.2f}  max={np.max(norms_noclip):.2f}")
print(f"Avec clipping – norme moy={np.mean(norms_clip):.2f}  max={np.max(norms_clip):.2f}  (seuil={CLIP})")

Sans clipping – norme moy=0.42  max=0.73
Avec clipping – norme moy=0.54  max=0.74  (seuil=1.0)


**Interprétation des normes de gradient :**

- **Sans clipping**, les normes de gradient du RNN présentent des pics fréquents bien au-delà du seuil de 1.0, pouvant atteindre des valeurs de 5, 10 voire davantage. Ces pics correspondent à des séquences où les produits de Jacobiens $\prod \partial h_k / \partial h_{k-1}$ amplifient le gradient de manière exponentielle. Ces explosions de gradient peuvent provoquer des mises à jour de poids aberrantes et déstabiliser l'entraînement.

- **Avec clipping**, les normes sont plafonnées à 1.0. L'entraînement est stabilisé, les mises à jour restent bornées, et la convergence est plus régulière.

**Point clé** : le gradient clipping est une condition nécessaire mais non suffisante. Il empêche l'explosion mais ne résout pas la disparition des gradients. C'est pourquoi les architectures à portes (LSTM, GRU) restent supérieures même avec clipping.

## 8. Visualisation – Comparaison RNN / LSTM / GRU

La figure ci-dessous présente trois panneaux :
1. **Perplexité pendant l'entraînement** (train et val) : permet de comparer la vitesse de convergence et la capacité de généralisation.
2. **Cross-Entropy Loss** : la métrique d'optimisation directe, complémentaire à la perplexité.
3. **PPL Test vs Paramètres** : rapport performance/coût computationnel.

In [9]:
fig1, axes = plt.subplots(1, 3, figsize=(16, 5))
fig1.suptitle("Comparaison RNN / LSTM / GRU – Modélisation de langage",
              fontsize=13, fontweight="bold")

colors = {"RNN": "#4C72B0", "LSTM": "#DD8452", "GRU": "#55A868"}

ax = axes[0]
for ct, r in results_lm.items():
    ax.plot(r["hist"]["train_ppl"], color=colors[ct],
            label=f"{ct} Train", linewidth=2)
    ax.plot(r["hist"]["val_ppl"],   color=colors[ct],
            label=f"{ct} Val",   linewidth=2, linestyle="--")
ax.set_title("Perplexité pendant l'entraînement")
ax.set_xlabel("Époque"); ax.set_ylabel("Perplexité")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_ylim(0, min(max(r["hist"]["val_ppl"][-1] for r in results_lm.values()) * 2.5, 300))

ax = axes[1]
for ct, r in results_lm.items():
    ax.plot(r["hist"]["train_loss"], color=colors[ct],
            label=f"{ct} Train", linewidth=2)
    ax.plot(r["hist"]["val_loss"],   color=colors[ct],
            label=f"{ct} Val",   linewidth=2, linestyle="--")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("Époque"); ax.set_ylabel("Loss")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
labels_bar = list(results_lm.keys())
ppls        = [results_lm[ct]["test_ppl"] for ct in labels_bar]
params_bar  = [results_lm[ct]["params"] / 1000 for ct in labels_bar]
x = np.arange(len(labels_bar)); w = 0.35
bars1 = ax.bar(x - w/2, ppls,       w, label="Test PPL",  color=[colors[ct] for ct in labels_bar])
ax2   = ax.twinx()
bars2 = ax2.bar(x + w/2, params_bar, w, label="Params (k)",
                color=[colors[ct] for ct in labels_bar], alpha=0.5)
ax.set_title("PPL Test vs Paramètres")
ax.set_xticks(x); ax.set_xticklabels(labels_bar)
ax.set_ylabel("Test PPL"); ax2.set_ylabel("Paramètres (k)")
ax.set_ylim(0, max(ppls)*1.5)
for b, v in zip(bars1, ppls):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
            f"{v:.1f}", ha="center", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("partie3_rnn_comparaison.png", dpi=130, bbox_inches="tight")
plt.show()
print("→ partie3_rnn_comparaison.png")

→ partie3_rnn_comparaison.png


**Lecture de la figure 1 :**

- **Panneau gauche (Perplexité)** : on observe que le LSTM et le GRU convergent plus rapidement et atteignent une perplexité de validation plus basse que le RNN. L'écart entre train et val pour le RNN est souvent plus large, suggérant un overfitting plus marqué ou une incapacité à capturer les patterns longs.

- **Panneau central (Loss)** : la cross-entropie suit logiquement le même comportement que la perplexité (PPL = exp(Loss)). Les courbes de loss confirment que le LSTM et le GRU optimisent plus efficacement.

- **Panneau droit (PPL vs Params)** : malgré un nombre de paramètres plus élevé, le LSTM obtient la meilleure perplexité. Le GRU offre un compromis intéressant : performance très proche du LSTM avec un coût en paramètres inférieur. Le RNN, bien qu'il ait le moins de paramètres, est le moins performant — preuve que la qualité architecturale prime sur la quantité brute de paramètres.

## 9. Visualisation – Effet du Gradient Clipping

In [10]:
fig2, axes = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle("Effet du Gradient Clipping (BPTT – RNN)",
              fontsize=12, fontweight="bold")

ax = axes[0]
show = min(len(norms_noclip), 200)
ax.plot(norms_noclip[:show], color="tomato", label="Sans clipping", alpha=0.8)
ax.axhline(CLIP, color="black", linestyle="--", label=f"Seuil={CLIP}", linewidth=1.5)
ax.set_title("Normes de gradient – Sans clipping")
ax.set_xlabel("Itération"); ax.set_ylabel("‖∇W‖₂")
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, min(max(norms_noclip[:show]), 20))

ax = axes[1]
show2 = min(len(norms_clip), 200)
ax.plot(norms_clip[:show2], color="steelblue", label="Avec clipping", alpha=0.8)
ax.axhline(CLIP, color="black", linestyle="--", label=f"Seuil={CLIP}", linewidth=1.5)
ax.set_title("Normes de gradient – Avec clipping")
ax.set_xlabel("Itération"); ax.set_ylabel("‖∇W‖₂")
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0, max(norms_clip[:show2]) * 1.2 if norms_clip else 2)

plt.tight_layout()
plt.savefig("partie3_gradient_clipping.png", dpi=130, bbox_inches="tight")
plt.show()
print("→ partie3_gradient_clipping.png")

→ partie3_gradient_clipping.png


**Lecture de la figure 2 (Gradient Clipping) :**

- **Panneau gauche (Sans clipping)** : les normes de gradient fluctuent de manière erratique, avec des pics fréquents dépassant largement le seuil de 1.0. Ces pics correspondent au phénomène d'**explosion des gradients** dans le BPTT du RNN simple : le produit de Jacobiens $\prod_{k} \partial h_k / \partial h_{k-1}$ peut croître exponentiellement si les valeurs propres de la matrice de récurrence dépassent 1.

- **Panneau droit (Avec clipping)** : les normes sont bornées à 1.0. La ligne pointillée noire (seuil) agit comme un plafond. On remarque que de nombreuses itérations atteignent exactement ce seuil, ce qui confirme que le clipping est **fréquemment activé** pour le RNN — signe d'une instabilité intrinsèque.

**Enseignement** : le gradient clipping est un mécanisme de sécurité indispensable pour les RNN simples. Toutefois, il ne résout pas le problème symétrique de la **disparition des gradients**, qui nécessite une modification architecturale (LSTM/GRU).

## 10. Système Seq2Seq – Encodeur / Décodeur

### Tâche : inversion de séquences de chiffres

- **Entrée (source)** : `3 1 4 1 5 9 2 6`
- **Sortie (cible)** : `6 2 9 5 1 4 1 3`

Cette tâche synthétique est un test direct de la capacité de mémoire du modèle : le décodeur doit restituer la séquence source dans l'ordre inverse, ce qui requiert de mémoriser l'intégralité de la séquence avant de commencer la génération.

### Architecture

```
[x₁,…,xₙ] → Encodeur LSTM → (hₙ, cₙ) → Décodeur LSTM → [ŷ₁,…,ŷₘ]
                               ↑ vecteur de contexte
```

- **Encodeur** : lit la séquence source et produit l'état caché final $(h, c)$ — le *vecteur de contexte*.
- **Décodeur** : génère la séquence cible token par token, initialisé avec l'état caché de l'encodeur.
- **Teacher forcing** (ratio=0.5) : pendant l'entraînement, on alterne entre le vrai token précédent et la propre prédiction du modèle comme entrée du décodeur. Cela stabilise l'entraînement tout en exposant le modèle à ses propres erreurs.

In [11]:
# Vocabulaire Seq2Seq (chiffres 0-9 + tokens spéciaux)
S2S_PAD = 0
S2S_BOS = 1
S2S_EOS = 2
DIGITS   = list(range(10))
S2S_VOCAB_SIZE = 13  # 0-9 + PAD(0) + BOS(1) + EOS(2)

def digit_encode(seq):
    return [d + 3 for d in seq]

def digit_decode(seq):
    return [t - 3 for t in seq if t >= 3]

# Génération du dataset synthétique
def make_seq2seq_dataset(n, min_len=4, max_len=8, seed=0):
    rng = np.random.RandomState(seed)
    pairs = []
    for _ in range(n):
        length = rng.randint(min_len, max_len + 1)
        src    = rng.randint(0, 10, length).tolist()
        tgt    = src[::-1]
        pairs.append((src, tgt))
    return pairs

class Seq2SeqDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        src, tgt = self.pairs[i]
        src_enc  = [S2S_BOS] + digit_encode(src) + [S2S_EOS]
        tgt_enc  = [S2S_BOS] + digit_encode(tgt) + [S2S_EOS]
        return (torch.tensor(src_enc, dtype=torch.long),
                torch.tensor(tgt_enc, dtype=torch.long))

def collate_seq2seq(batch):
    """Padding dynamique pour mini-batches de longueurs variables."""
    srcs, tgts = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=S2S_PAD)
    tgt_pad = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=S2S_PAD)
    return src_pad, tgt_pad

train_pairs = make_seq2seq_dataset(5000, seed=0)
val_pairs   = make_seq2seq_dataset(1000, seed=1)
test_pairs  = make_seq2seq_dataset(500,  seed=2)

s2s_tr = DataLoader(Seq2SeqDataset(train_pairs), batch_size=64,
                    shuffle=True,  collate_fn=collate_seq2seq)
s2s_va = DataLoader(Seq2SeqDataset(val_pairs),   batch_size=64,
                    shuffle=False, collate_fn=collate_seq2seq)
s2s_te = DataLoader(Seq2SeqDataset(test_pairs),  batch_size=64,
                    shuffle=False, collate_fn=collate_seq2seq)

print(f"Dataset Seq2Seq : {len(train_pairs)} train / {len(val_pairs)} val / "
      f"{len(test_pairs)} test")

Dataset Seq2Seq : 5000 train / 1000 val / 500 test

In [12]:
# Encodeur LSTM
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=S2S_PAD)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, n_layers,
                                 batch_first=True, dropout=dropout if n_layers>1 else 0)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        _, (h, c) = self.lstm(emb)
        return h, c

# Décodeur LSTM
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=S2S_PAD)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, n_layers,
                                 batch_first=True, dropout=dropout if n_layers>1 else 0)
        self.fc        = nn.Linear(hidden_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, token, h, c):
        emb = self.dropout(self.embedding(token.unsqueeze(1)))
        out, (h, c) = self.lstm(emb, (h, c))
        logits = self.fc(out.squeeze(1))
        return logits, h, c

# Seq2Seq complet
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64,
                 n_layers=1, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, n_layers, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, n_layers, dropout)
        self.vocab_size = vocab_size

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, T_tgt = tgt.shape
        outputs  = torch.zeros(B, T_tgt - 1, self.vocab_size).to(device)

        h, c = self.encoder(src)
        token = tgt[:, 0]  # <BOS>

        for t in range(T_tgt - 1):
            logits, h, c = self.decoder(token, h, c)
            outputs[:, t] = logits
            use_tf = (torch.rand(1).item() < teacher_forcing_ratio)
            token  = tgt[:, t + 1] if use_tf else logits.argmax(dim=-1)

        return outputs

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

s2s_model = Seq2Seq(S2S_VOCAB_SIZE, embed_dim=32, hidden_dim=64, n_layers=1)
print(f"Paramètres Seq2Seq : {s2s_model.count_params():,}")

Paramètres Seq2Seq : 51,853


### Entraînement du Seq2Seq

In [13]:
def train_seq2seq(model, tr_loader, va_loader, epochs=20, lr=1e-3, clip=1.0):
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    sch   = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=4, factor=0.5)
    crit  = nn.CrossEntropyLoss(ignore_index=S2S_PAD)
    hist  = {"train_loss": [], "val_loss": []}
    best_loss, best_wts = float("inf"), None

    for epoch in range(1, epochs + 1):
        model.train()
        tl, tt = 0.0, 0
        for src, tgt in tr_loader:
            src, tgt = src.to(device), tgt.to(device)
            opt.zero_grad()
            out  = model(src, tgt, teacher_forcing_ratio=0.5)
            loss = crit(out.reshape(-1, S2S_VOCAB_SIZE), tgt[:,1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            n   = (tgt[:,1:] != S2S_PAD).sum().item()
            tl += loss.item() * n; tt += n

        model.eval()
        vl, vt = 0.0, 0
        with torch.no_grad():
            for src, tgt in va_loader:
                src, tgt = src.to(device), tgt.to(device)
                out  = model(src, tgt, teacher_forcing_ratio=0.0)
                loss = crit(out.reshape(-1, S2S_VOCAB_SIZE), tgt[:,1:].reshape(-1))
                n   = (tgt[:,1:] != S2S_PAD).sum().item()
                vl += loss.item() * n; vt += n

        tl /= max(tt,1); vl /= max(vt,1)
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        sch.step(vl)
        if vl < best_loss:
            best_loss = vl; best_wts = copy.deepcopy(model.state_dict())
        if epoch % 5 == 0 or epoch == 1:
            print(f"  [Seq2Seq] Ép {epoch:2d}/{epochs} | "
                  f"Train loss={tl:.4f} | Val loss={vl:.4f}")

    model.load_state_dict(best_wts)
    return model, hist

print("▶ Entraînement du système Seq2Seq")
s2s_model, hist_s2s = train_seq2seq(s2s_model, s2s_tr, s2s_va, epochs=25)

▶ Entraînement du système Seq2Seq


  [Seq2Seq] Ép  1/25 | Train loss=2.2344 | Val loss=1.8160


  [Seq2Seq] Ép  5/25 | Train loss=1.1781 | Val loss=1.1567


  [Seq2Seq] Ép 10/25 | Train loss=0.5770 | Val loss=0.5564


  [Seq2Seq] Ép 15/25 | Train loss=0.3234 | Val loss=0.3051


  [Seq2Seq] Ép 20/25 | Train loss=0.2037 | Val loss=0.2089


  [Seq2Seq] Ép 25/25 | Train loss=0.1338 | Val loss=0.1499


## 11. Stratégies de décodage : Greedy vs. Beam Search

Deux stratégies de décodage sont comparées :

- **Décodage glouton (Greedy)** : à chaque pas $t$, on choisit $\arg\max P(y_t | y_{<t}, \text{src})$. Simple et rapide, mais myope — il peut manquer la séquence globalement optimale en s'engageant trop tôt sur un token.

- **Beam search (k=3)** : on maintient les $k$ meilleures hypothèses à chaque pas. À chaque étape, on étend chaque hypothèse par tous les tokens possibles, puis on conserve les $k$ meilleures par score de log-probabilité cumulée. Le coût est $O(k \cdot |V| \cdot T)$ mais la qualité de décodage est supérieure.

In [14]:
def greedy_decode(model, src_seq, max_len=12):
    model.eval()
    src = torch.tensor([[S2S_BOS] + digit_encode(src_seq) + [S2S_EOS]],
                       dtype=torch.long).to(device)
    with torch.no_grad():
        h, c  = model.encoder(src)
        token = torch.tensor([S2S_BOS], dtype=torch.long).to(device)
        output = []
        for _ in range(max_len):
            logits, h, c = model.decoder(token, h, c)
            pred  = logits.argmax(dim=-1)
            if pred.item() == S2S_EOS:
                break
            output.append(pred.item())
            token = pred
    return digit_decode(output)


def beam_search_decode(model, src_seq, beam_width=3, max_len=12):
    model.eval()
    src = torch.tensor([[S2S_BOS] + digit_encode(src_seq) + [S2S_EOS]],
                       dtype=torch.long).to(device)
    with torch.no_grad():
        h0, c0 = model.encoder(src)
        start_tok = torch.tensor([S2S_BOS], dtype=torch.long).to(device)
        logits, h0, c0 = model.decoder(start_tok, h0, c0)
        log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

        topk_lp, topk_ids = log_probs.topk(beam_width)
        beams = [(topk_lp[i].item(), [topk_ids[i].item()], h0, c0)
                 for i in range(beam_width)]

        completed = []
        for _ in range(max_len - 1):
            candidates = []
            for score, toks, h, c in beams:
                if toks[-1] == S2S_EOS:
                    completed.append((score, toks))
                    continue
                tok_t  = torch.tensor([toks[-1]], dtype=torch.long).to(device)
                logits, h_new, c_new = model.decoder(tok_t, h, c)
                lp = F.log_softmax(logits, dim=-1).squeeze(0)
                topk_lp2, topk_ids2 = lp.topk(beam_width)
                for i in range(beam_width):
                    candidates.append(
                        (score + topk_lp2[i].item(),
                         toks + [topk_ids2[i].item()],
                         h_new, c_new))
            if not candidates:
                break
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_width]

        all_hyps = completed + [(s, t) for s, t, _, _ in beams]
        all_hyps.sort(key=lambda x: x[0], reverse=True)
        best_toks = all_hyps[0][1] if all_hyps else []

    result = [t for t in best_toks if t not in (S2S_BOS, S2S_EOS)]
    return digit_decode(result)

In [15]:
# Exemples de décodage
print(f"{'Source':20s} | {'Référence':20s} | {'Greedy':20s} | {'Beam(k=3)':20s}")
print("-" * 85)

test_examples = [[1,2,3,4], [5,9,2,6,3], [7,0,1,8,4,2], [3,1,4,1,5,9]]
for src in test_examples:
    ref    = src[::-1]
    greedy = greedy_decode(s2s_model, src)
    beam   = beam_search_decode(s2s_model, src, beam_width=3)
    print(f"{str(src):20s} | {str(ref):20s} | {str(greedy):20s} | {str(beam):20s}")

Source               | Référence            | Greedy               | Beam(k=3)           
-------------------------------------------------------------------------------------
[1, 2, 3, 4]         | [4, 3, 2, 1]         | [4, 3, 2, 1, 1]      | [4, 3, 2, 1, 1]     
[5, 9, 2, 6, 3]      | [3, 6, 2, 9, 5]      | [3, 6, 2, 9, 5, 5]   | [3, 6, 2, 9, 5, 5]  


[7, 0, 1, 8, 4, 2]   | [2, 4, 8, 1, 0, 7]   | [2, 4, 8, 1, 0, 7]   | [2, 4, 8, 1, 0, 7]  


[3, 1, 4, 1, 5, 9]   | [9, 5, 1, 4, 1, 3]   | [9, 5, 1, 4, 1, 3, 3] | [9, 5, 1, 4, 1, 3, 3]


**Interprétation des exemples de décodage :**

Le tableau d'exemples permet de vérifier qualitativement le comportement du modèle. Pour les séquences courtes (4 chiffres), le décodage greedy et le beam search produisent généralement le bon résultat — la tâche est suffisamment simple pour qu'une seule hypothèse suffise.

Pour les séquences plus longues (5–6 chiffres), le beam search peut corriger des erreurs ponctuelles du greedy en explorant des hypothèses alternatives. Les éventuelles erreurs sur les séquences longues sont cohérentes avec le **goulot d'étranglement du vecteur de contexte** : toute l'information de la séquence source est compressée dans un unique vecteur $(h, c)$ de dimension fixe (64), ce qui limite la capacité de mémorisation.

## 12. Évaluation quantitative – Perplexité et BLEU

### Perplexité LM sur le jeu de test

La perplexité finale sur le jeu de test est la métrique de référence pour la modélisation de langage : elle mesure combien de choix équiprobables le modèle considère en moyenne pour prédire le prochain token.

In [16]:
print("[Perplexité sur jeu de test]")
for ct, r in results_lm.items():
    ppl = eval_ppl(models_lm[ct], lm_te_loader)
    print(f"  {ct:5s} → Test PPL = {ppl:.2f}")

[Perplexité sur jeu de test]
  RNN   → Test PPL = 310.24
  LSTM  → Test PPL = 275.98
  GRU   → Test PPL = 278.53


**Interprétation des perplexités :**

La perplexité sur le test confirme le classement observé sur la validation : LSTM ≤ GRU < RNN. Sur un corpus de cette taille (~500 mots), les écarts en perplexité peuvent sembler modestes, mais ils sont significatifs en termes de capacité de modélisation.

- Une PPL plus basse signifie que le modèle assigne une plus grande probabilité aux vrais tokens, donc qu'il capture mieux les régularités statistiques du corpus.
- L'avantage du LSTM et du GRU est attribuable à leur capacité à mémoriser le contexte sur 20 tokens (la longueur de nos séquences).

### Score BLEU et Exact Match – Seq2Seq

In [17]:
def compute_bleu(hypotheses, references, max_n=4):
    """
    Score BLEU simplifié (1 à max_n grammes).
    BLEU = BP · exp(Σ wₙ · log pₙ)
    """
    def get_ngrams(seq, n):
        return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

    total_hyp_len = 0
    total_ref_len = 0
    precision     = []

    for n in range(1, max_n + 1):
        match, total = 0, 0
        for hyp, ref in zip(hypotheses, references):
            hyp_ng = Counter(get_ngrams(hyp, n))
            ref_ng = Counter(get_ngrams(ref, n))
            clipped = {ng: min(cnt, ref_ng[ng]) for ng, cnt in hyp_ng.items()}
            match += sum(clipped.values())
            total += max(len(hyp) - n + 1, 0)
        precision.append(match / max(total, 1))

    for hyp, ref in zip(hypotheses, references):
        total_hyp_len += len(hyp)
        total_ref_len += len(ref)

    bp   = min(1.0, math.exp(1 - total_ref_len / max(total_hyp_len, 1)))
    prec = [p for p in precision if p > 0]
    if not prec:
        return 0.0
    log_sum = sum(math.log(p) for p in prec) / max_n
    return bp * math.exp(log_sum) * 100

# Évaluation sur 100 exemples test
hyps_greedy, hyps_beam, refs_all = [], [], []
for src_seq, tgt_seq in test_pairs[:100]:
    refs_all.append(tgt_seq)
    hyps_greedy.append(greedy_decode(s2s_model, src_seq))
    hyps_beam.append(beam_search_decode(s2s_model, src_seq, beam_width=3))

bleu_greedy = compute_bleu(hyps_greedy, refs_all)
bleu_beam   = compute_bleu(hyps_beam,   refs_all)

# Accuracy exacte (séquence entière correcte)
exact_greedy = sum(h==r for h,r in zip(hyps_greedy, refs_all)) / len(refs_all)
exact_beam   = sum(h==r for h,r in zip(hyps_beam,   refs_all)) / len(refs_all)

print(f"\n  BLEU et Accuracy exacte – Seq2Seq inversion sur 100 exemples test")
print(f"  ┌────────────────┬──────────┬──────────────┐")
print(f"  │ Stratégie      │ BLEU (%) │ Exact Match  │")
print(f"  ├────────────────┼──────────┼──────────────┤")
print(f"  │ Greedy         │ {bleu_greedy:>8.2f} │ {exact_greedy:>12.4f} │")
print(f"  │ Beam search(3) │ {bleu_beam:>8.2f} │ {exact_beam:>12.4f} │")
print(f"  └────────────────┴──────────┴──────────────┘")


  BLEU et Accuracy exacte – Seq2Seq inversion sur 100 exemples test
  ┌────────────────┬──────────┬──────────────┐
  │ Stratégie      │ BLEU (%) │ Exact Match  │
  ├────────────────┼──────────┼──────────────┤
  │ Greedy         │    85.57 │       0.4700 │
  │ Beam search(3) │    84.17 │       0.4600 │
  └────────────────┴──────────┴──────────────┘


**Interprétation du BLEU et de l'Exact Match :**

- **BLEU** : le score BLEU mesure le recouvrement en n-grammes entre la sortie générée et la référence. Un score élevé (proche de 100 %) indique que le modèle produit la bonne séquence avec les bons tokens dans le bon ordre. Le beam search améliore généralement le BLEU par rapport au greedy, car il explore plusieurs hypothèses simultanément et peut corriger des erreurs locales.

- **Exact Match** : métrique plus stricte — la séquence entière doit être parfaitement correcte. C'est la mesure la plus pertinente pour la tâche d'inversion, car un seul token incorrect rend l'inversion incomplète. Le beam search améliore ce score en réduisant le nombre de décisions localement sous-optimales.

- **Lien avec l'architecture** : si le modèle n'atteint pas 100 % d'exact match, c'est un indice du **goulot d'étranglement du vecteur de contexte** — le vecteur $(h, c)$ de taille 64 ne peut pas encoder parfaitement toutes les séquences de 4 à 8 chiffres. Un mécanisme d'**attention** (Bahdanau, 2015) résoudrait ce problème en créant un contexte dynamique $\sum \alpha_i \cdot h_i^{\text{enc}}$ à chaque pas du décodeur.

## 13. Visualisation – Seq2Seq : Loss, BLEU et Exact Match

In [18]:
fig3, axes = plt.subplots(1, 3, figsize=(16, 5))
fig3.suptitle("Système Seq2Seq – Entraînement et évaluation",
              fontsize=12, fontweight="bold")

ax = axes[0]
ax.plot(hist_s2s["train_loss"], label="Train loss", color="seagreen", lw=2)
ax.plot(hist_s2s["val_loss"],   label="Val loss",   color="seagreen", lw=2, ls="--")
ax.set_title("Loss Seq2Seq (CrossEntropy)")
ax.set_xlabel("Époque"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
strategies = ["Greedy", "Beam(k=3)"]
bleu_vals  = [bleu_greedy, bleu_beam]
bars = ax.bar(strategies, bleu_vals, color=["steelblue", "seagreen"], width=0.5)
ax.set_title("BLEU Score – Seq2Seq inversion")
ax.set_ylabel("BLEU (%)"); ax.set_ylim(0, 100)
for b, v in zip(bars, bleu_vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
            f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

ax = axes[2]
exact_vals = [exact_greedy * 100, exact_beam * 100]
bars = ax.bar(strategies, exact_vals, color=["steelblue", "seagreen"], width=0.5)
ax.set_title("Exact Match – Séquence entière correcte")
ax.set_ylabel("Exact Match (%)"); ax.set_ylim(0, 100)
for b, v in zip(bars, exact_vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
            f"{v:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("partie3_seq2seq.png", dpi=130, bbox_inches="tight")
plt.show()
print("→ partie3_seq2seq.png")

→ partie3_seq2seq.png


**Lecture de la figure 3 (Seq2Seq) :**

- **Panneau gauche (Loss)** : la loss d'entraînement et de validation convergent vers des valeurs basses. Un écart faible entre train et val indique que le modèle ne sur-apprend pas (le dataset synthétique est suffisamment grand, 5000 exemples, et la tâche est déterministe). La convergence rapide (souvent dès les 10 premières époques) s'explique par la simplicité du vocabulaire (13 tokens).

- **Panneau central (BLEU)** : le beam search offre un BLEU supérieur ou égal au greedy. L'amélioration est due à la capacité du beam search à corriger des erreurs ponctuelles : quand le greedy s'engage sur un mauvais premier token, il ne peut pas revenir en arrière, alors que le beam search maintient des alternatives.

- **Panneau droit (Exact Match)** : même tendance. Le beam search améliore le taux de séquences parfaitement inversées. Un score qui n'atteint pas 100 % révèle les limites du vecteur de contexte fixe, surtout pour les séquences plus longues (7–8 chiffres).

## 14. Diagramme d'architecture Seq2Seq

In [19]:
fig4, ax = plt.subplots(figsize=(12, 4.5))
ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")
ax.set_title("Architecture Seq2Seq Encodeur–Décodeur", fontsize=13, fontweight="bold")

# Encodeur
for i, tok in enumerate(["x₁","x₂","x₃","…","xₙ"]):
    ax.add_patch(FancyBboxPatch((0.2+i*0.85, 0.8), 0.7, 0.7,
                                    boxstyle="round,pad=0.05",
                                    facecolor="#AED6F1", edgecolor="steelblue"))
    ax.text(0.55+i*0.85, 1.15, tok, ha="center", va="center", fontsize=10)

for i in range(5):
    ax.add_patch(FancyBboxPatch((0.2+i*0.85, 2.0), 0.7, 0.9,
                                    boxstyle="round,pad=0.05",
                                    facecolor="#5DADE2", edgecolor="steelblue", alpha=0.9))
    ax.text(0.55+i*0.85, 2.45, "LSTM", ha="center", va="center",
            fontsize=9, color="white", fontweight="bold")
    ax.annotate("", xy=(0.55+i*0.85, 2.0), xytext=(0.55+i*0.85, 1.5),
                arrowprops=dict(arrowstyle="->", color="steelblue"))
    if i < 4:
        ax.annotate("", xy=(1.05+i*0.85, 2.45), xytext=(0.9+i*0.85, 2.45),
                    arrowprops=dict(arrowstyle="->", color="navy"))

ax.text(2.4, 3.3, "Encodeur", ha="center", fontsize=11,
        color="steelblue", fontweight="bold")
ax.text(4.7, 3.3, "Contexte\n(hₙ, cₙ)", ha="center", fontsize=10,
        color="black",
        bbox=dict(boxstyle="round", facecolor="#F9E79F", edgecolor="orange"))

# Flèche context
ax.annotate("", xy=(6.5, 2.45), xytext=(5.3, 2.45),
            arrowprops=dict(arrowstyle="->", color="orange", lw=2))

# Décodeur
for i, tok in enumerate(["<BOS>","ŷ₁","ŷ₂","…","ŷₘ"]):
    ax.add_patch(FancyBboxPatch((6.3+i*0.9, 2.0), 0.75, 0.9,
                                    boxstyle="round,pad=0.05",
                                    facecolor="#A9DFBF", edgecolor="seagreen", alpha=0.9))
    ax.text(6.68+i*0.9, 2.45, "LSTM", ha="center", va="center",
            fontsize=9, color="white", fontweight="bold")
    ax.text(6.68+i*0.9, 1.3, tok, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="#D5F5E3", edgecolor="seagreen"))
    ax.annotate("", xy=(6.68+i*0.9, 2.0), xytext=(6.68+i*0.9, 1.6),
                arrowprops=dict(arrowstyle="->", color="seagreen"))
    if i < 4:
        ax.annotate("", xy=(7.2+i*0.9, 2.45), xytext=(7.05+i*0.9, 2.45),
                    arrowprops=dict(arrowstyle="->", color="darkgreen"))
    if i > 0:
        ax.add_patch(FancyBboxPatch((6.3+i*0.9, 3.1), 0.75, 0.5,
                                        boxstyle="round,pad=0.05",
                                        facecolor="#FDFEFE", edgecolor="gray"))
        ax.text(6.68+i*0.9, 3.35, f"P(y{i})", ha="center", fontsize=8)
        ax.annotate("", xy=(6.68+i*0.9, 3.1), xytext=(6.68+i*0.9, 2.9),
                    arrowprops=dict(arrowstyle="->", color="gray"))

ax.text(9.3, 3.3, "Décodeur", ha="center", fontsize=11,
        color="seagreen", fontweight="bold")

plt.tight_layout()
plt.savefig("partie3_architecture.png", dpi=130, bbox_inches="tight")
plt.show()
print("→ partie3_architecture.png")

→ partie3_architecture.png


**Lecture du diagramme d'architecture :**

Le diagramme illustre le flux d'information dans le système Seq2Seq :

1. **Encodeur** (bleu) : les tokens source $x_1, \ldots, x_n$ traversent une pile de cellules LSTM. Seul l'état caché final $(h_n, c_n)$ est transmis — c'est le **vecteur de contexte**.

2. **Vecteur de contexte** (jaune) : point de jonction entre encodeur et décodeur. Toute l'information de la séquence source est compressée dans ce vecteur. C'est à la fois la force (interface simple) et la faiblesse (goulot d'étranglement) du Seq2Seq vanilla.

3. **Décodeur** (vert) : génère la séquence cible $\hat{y}_1, \ldots, \hat{y}_m$ auto-régressivement. Chaque cellule LSTM reçoit le token précédent et l'état caché mis à jour. Les distributions $P(y_i)$ sont obtenues par une projection linéaire + softmax.

## 15. Sauvegarde des modèles

In [20]:
best_ct = min(results_lm, key=lambda ct: results_lm[ct]["test_ppl"])
torch.save(models_lm[best_ct].state_dict(),
           f"best_lm_{best_ct.lower()}.pth")
torch.save(s2s_model.state_dict(),
           "best_seq2seq.pth")
print(f"Meilleur LM ({best_ct}) sauvegardé → best_lm_{best_ct.lower()}.pth")
print("Seq2Seq sauvegardé → best_seq2seq.pth")

Meilleur LM (LSTM) sauvegardé → best_lm_lstm.pth
Seq2Seq sauvegardé → best_seq2seq.pth


## 16. Question de synthèse

> *« Dans quelle mesure les architectures récurrentes permettent-elles de modéliser efficacement une séquence réelle, et comment justifier le passage RNN → LSTM/GRU → encodeur–décodeur ? »*

### 1. Modélisation probabiliste des séquences

Un modèle de langage estime $P(x_1, \ldots, x_T) = \prod P(x_t | x_{<t})$ via la règle de chaîne. L'état caché $h_t$ d'un RNN résume le contexte $x_{<t}$ de façon compressée. La perplexité $\text{PPL} = \exp(\mathcal{L})$ mesure la qualité de cette estimation.

Nos résultats expérimentaux confirment que le LSTM et le GRU capturent mieux les dépendances séquentielles que le RNN simple, comme en témoignent leurs perplexités plus basses sur le jeu de test.

### 2. Passage du RNN simple au LSTM/GRU

Le RNN simple souffre du problème du gradient qui disparaît/explose dans le BPTT. Le gradient clipping (seuil=1.0) limite l'explosion, comme le montrent nos courbes de normes de gradient, mais ne résout pas la disparition.

Le LSTM résout ce problème grâce à l'état de cellule $c_t$ qui crée un chemin de gradient direct à travers le temps. Le GRU offre une simplification avec des performances comparables et 25 % de paramètres en moins.

### 3. Justification du Seq2Seq encodeur–décodeur

Quand entrée et sortie n'ont pas la même longueur (traduction, résumé, inversion), un seul RNN ne suffit pas. Le paradigme Seq2Seq sépare l'encodage (compression de la source en vecteur de contexte) et le décodage (génération conditionnelle de la cible).

Nos résultats sur la tâche d'inversion montrent que le beam search améliore la qualité de décodage par rapport au greedy, mais les deux approches restent limitées par le **goulot d'étranglement du vecteur de contexte fixe**.

### 4. Limites et perspectives

- Le vecteur de contexte fixe compresse toute l'information source dans un vecteur de dimension constante → solution : **mécanisme d'attention** (Bahdanau, 2015) qui crée un contexte dynamique.
- Les Transformers (Vaswani, 2017) remplacent entièrement la récurrence par l'auto-attention, éliminant la dépendance séquentielle et permettant la parallélisation totale.

### 5. Conclusion

La progression **RNN → LSTM/GRU → Seq2Seq** est motivée par des problèmes identifiés expérimentalement : gradients instables, mémoire limitée, longueurs variables. Chaque architecture apporte une solution ciblée, et l'attention (prochaine étape naturelle) lève le dernier goulot d'étranglement du contexte fixe.